# 8.Agent的高级用法4：流式输出及模式

## 8.1 流式输出的说明
通过 invoke 调用Agent时，内部可能经历多次调用，长时间看不到调用情况，用户体验不好，可以通过 流式调用 （渐进式显示输出）优化用户体验， 实时显示 Agent 运行过程中的更新。特别是在处理LLM 延迟时尤其有效。

流式输出好处：
* 大型语言模型生成完整响应通常需要几秒钟时间，对于长输出可能达到10-20 秒，用户期望即时反馈， 流式传输让等待过程更加可控 。
* 相比非流式传输需要用户长时间等待完整响应，流式传输可以立即显示文字逐渐出现的效果， 大幅降低用户的等待焦虑 。

设置方式：
通过`agent.stream(stream_mode=指定模式)`来指定。具体模式有：values、updates（默认）、messages、custom、checkpoints、tasks、debug。

## 8.2 具体的输出模式
### 8.2.1 values输出模式
当 stream_mode 设置为values模式时，每个步骤执行后，都会输出完整的状态信息，适用于每一步都要获取完整状态、状态持久化场景。

In [4]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
)

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint
@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息
    
    Args:
        customer_id: 客户ID，用于唯一标识客户
    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等"""
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}

@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史
    
    Args:
        customer_id: 客户ID，用于唯一标识客户
    Returns:
        包含客户订单历史的字典，如总订单数、总花费等"""
    return {"total_orders": 15,"total_spent": 25800.00}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动
    
    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等"""
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }

# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user", "content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
        stream_mode="values"
    ):
    rprint(chunk)
    print("-" * 50)

{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='f52b0624-7038-4767-978e-56518b5f7aee'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='f52b0624-7038-4767-978e-56518b5f7aee'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"customer_id":"CUST123456"}',
                    'call_id': 'call_b1fouNkXJp43i0TfCtTUpOH1',
                    'name': 'query_customer_data',
                    'type': 'function_call',
                    'id': 'fc_0e4d92fd88edfe9a016ac301ae263087d287544cd8da486c57',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'
                    },
                    'metadata': {'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'}
                },
                {
                    'arguments': '{"customer_id":"CUST123456"}',
                    'call_id': 'call_O81Ytb1Qvtr2ZiY29peZ1bFM',
                    'name': 'check_order_history',
                    'type': 'function_call',
                    'id': 'fc_0e4d92fd88edfe9a016ac301ae264487d2b2e697073b040b2c',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'
                    },
                    'metadata': {'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'}
                },
                {
                    'arguments': '{}',
                    'call_id': 'call_C4pgqa0vTCUN1nglN5YL1jp2',
                    'name': 'get_current_promotions',
                    'type': 'function_call',
                    'id': 'fc_0e4d92fd88edfe9a016ac301ae264c87d28a46911c06a3f846',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'
                    },
                    'metadata': {'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'}
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0e4d92fd88edfe9a016ac301ab25f887d2890c137036b32bef',
                'created_at': 1791164843.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0e4d92fd88edfe9a016ac301ab25f887d2890c137036b32bef',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_b1fouNkXJp43i0TfCtTUpOH1',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_O81Ytb1Qvtr2ZiY29peZ1bFM',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_C4pgqa0vTCUN1nglN5YL1jp2',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4577,
                'output_tokens': 72,
                'total_tokens': 4649,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='f52b0624-7038-4767-978e-56518b5f7aee'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"customer_id":"CUST123456"}',
                    'call_id': 'call_b1fouNkXJp43i0TfCtTUpOH1',
                    'name': 'query_customer_data',
                    'type': 'function_call',
                    'id': 'fc_0e4d92fd88edfe9a016ac301ae263087d287544cd8da486c57',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'
                    },
                    'metadata': {'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'}
                },
                {
                    'arguments': '{"customer_id":"CUST123456"}',
                    'call_id': 'call_O81Ytb1Qvtr2ZiY29peZ1bFM',
                    'name': 'check_order_history',
                    'type': 'function_call',
                    'id': 'fc_0e4d92fd88edfe9a016ac301ae264487d2b2e697073b040b2c',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'
                    },
                    'metadata': {'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'}
                },
                {
                    'arguments': '{}',
                    'call_id': 'call_C4pgqa0vTCUN1nglN5YL1jp2',
                    'name': 'get_current_promotions',
                    'type': 'function_call',
                    'id': 'fc_0e4d92fd88edfe9a016ac301ae264c87d28a46911c06a3f846',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'
                    },
                    'metadata': {'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'}
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0e4d92fd88edfe9a016ac301ab25f887d2890c137036b32bef',
                'created_at': 1791164843.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0e4d92fd88edfe9a016ac301ab25f887d2890c137036b32bef',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_b1fouNkXJp43i0TfCtTUpOH1',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_O81Ytb1Qvtr2ZiY29peZ1bFM',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_C4pgqa0vTCUN1nglN5YL1jp2',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4577,
                'output_tokens': 72,
                'total_tokens': 4649,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='ada41c15-7421-412

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='f52b0624-7038-4767-978e-56518b5f7aee'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"customer_id":"CUST123456"}',
                    'call_id': 'call_b1fouNkXJp43i0TfCtTUpOH1',
                    'name': 'query_customer_data',
                    'type': 'function_call',
                    'id': 'fc_0e4d92fd88edfe9a016ac301ae263087d287544cd8da486c57',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'
                    },
                    'metadata': {'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'}
                },
                {
                    'arguments': '{"customer_id":"CUST123456"}',
                    'call_id': 'call_O81Ytb1Qvtr2ZiY29peZ1bFM',
                    'name': 'check_order_history',
                    'type': 'function_call',
                    'id': 'fc_0e4d92fd88edfe9a016ac301ae264487d2b2e697073b040b2c',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'
                    },
                    'metadata': {'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'}
                },
                {
                    'arguments': '{}',
                    'call_id': 'call_C4pgqa0vTCUN1nglN5YL1jp2',
                    'name': 'get_current_promotions',
                    'type': 'function_call',
                    'id': 'fc_0e4d92fd88edfe9a016ac301ae264c87d28a46911c06a3f846',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'
                    },
                    'metadata': {'turn_id': '01a109be-7fe8-736c-a3d0-a9a465213539'}
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0e4d92fd88edfe9a016ac301ab25f887d2890c137036b32bef',
                'created_at': 1791164843.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0e4d92fd88edfe9a016ac301ab25f887d2890c137036b32bef',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_b1fouNkXJp43i0TfCtTUpOH1',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_O81Ytb1Qvtr2ZiY29peZ1bFM',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_C4pgqa0vTCUN1nglN5YL1jp2',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4577,
                'output_tokens': 72,
                'total_tokens': 4649,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='ada41c15-7421-412

--------------------------------------------------


### 8.2.2 updates输出模式
这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。

代码如下：

In [6]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},stream_mode="updates"
):
    rprint(chunk)
    print("-" * 50)

{
    'model': {
        'messages': [
            AIMessage(
                content=[
                    {
                        'arguments': '{"customer_id":"CUST123456"}',
                        'call_id': 'call_RZQvAjAjHk66EAYhKY7cDNxb',
                        'name': 'query_customer_data',
                        'type': 'function_call',
                        'id': 'fc_0e3ed32166dd5d56016ac301ca71a087d28d0b6e99abc3b52e',
                        'status': 'completed'
                    },
                    {
                        'arguments': '{"customer_id":"CUST123456"}',
                        'call_id': 'call_nBI3PZedzzD7QUU7MBQFzFpF',
                        'name': 'check_order_history',
                        'type': 'function_call',
                        'id': 'fc_0e3ed32166dd5d56016ac301ca71ac87d28c4327759e87d4c0',
                        'status': 'completed'
                    },
                    {
                        'arguments': '{}',
                        'call_id': 'call_kN6uZR4529gCPp7FJwYZ73CB',
                        'name': 'get_current_promotions',
                        'type': 'function_call',
                        'id': 'fc_0e3ed32166dd5d56016ac301ca71b487d28a4121d66e467c93',
                        'status': 'completed'
                    }
                ],
                additional_kwargs={},
                response_metadata={
                    'id': 'resp_0e3ed32166dd5d56016ac301c8b64c87d29761ea035d4ead71',
                    'created_at': 1791164872.0,
                    'metadata': {},
                    'model': 'gpt-6-luna',
                    'object': 'response',
                    'service_tier': 'default',
                    'status': 'completed',
                    'model_provider': 'openai',
                    'model_name': 'gpt-6-luna'
                },
                id='resp_0e3ed32166dd5d56016ac301c8b64c87d29761ea035d4ead71',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_RZQvAjAjHk66EAYhKY7cDNxb',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_nBI3PZedzzD7QUU7MBQFzFpF',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_kN6uZR4529gCPp7FJwYZ73CB',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 4574,
                    'output_tokens': 72,
                    'total_tokens': 4646,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                    'output_token_details': {'reasoning': 0}
                }
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='81039381-37f0-4c83-b55c-b974574587dc',
                tool_call_id='call_RZQvAjAjHk66EAYhKY7cDNxb'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='8126e6ee-c044-4838-8888-3d8da0b9de32',
                tool_call_id='call_nBI3PZedzzD7QUU7MBQFzFpF'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
                name='get_current_promotions',
                id='fd36dd1b-f95d-413d-aa2c-f00de56c8a94',
                tool_call_id='call_kN6uZR4529gCPp7FJwYZ73CB'
            )
        ]
    }
}

--------------------------------------------------


{
    'model': {
        'messages': [
            AIMessage(
                content=[
                    {
                        'type': 'text',
                        'text': '客户 CUST123456 的信息如下：\n\n- 姓名：张三\n- 等级：VIP\n- 
加入日期：2023-01-15\n- 历史订单数：15 笔\n- 
历史消费总额：25,800.00\n\n当前可用优惠：老用户优惠、会员专属折扣，有效期至 2027-01-31。',
                        'annotations': [],
                        'id': 'msg_0e3ed32166dd5d56016ac301ce2ff887d28dd41a058d023b1f',
                        'phase': 'final_answer'
                    }
                ],
                additional_kwargs={},
                response_metadata={
                    'id': 'resp_0e3ed32166dd5d56016ac301cce39c87d28e829e1a0715d930',
                    'created_at': 1791164877.0,
                    'metadata': {},
                    'model': 'gpt-6-luna',
                    'object': 'response',
                    'service_tier': 'default',
                    'status': 'completed',
                    'model_provider': 'openai',
                    'model_name': 'gpt-6-luna'
                },
                id='resp_0e3ed32166dd5d56016ac301cce39c87d28e829e1a0715d930',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 4750,
                    'output_tokens': 91,
                    'total_tokens': 4841,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                    'output_token_details': {'reasoning': 0}
                }
            )
        ]
    }
}

--------------------------------------------------


### 8.2.3 messages输出模式
该模式中会输出流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现类似ChatGPT 的打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。


In [7]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},stream_mode="messages"
):
    rprint(chunk)
    print("-" * 50)

(
    AIMessageChunk(
        content=[],
        additional_kwargs={},
        response_metadata={
            'model_provider': 'openai',
            'id': 'resp_0f528ac9bcba3512016ac303e55a4087d2914bc240a80db67f'
        },
        id='resp_0f528ac9bcba3512016ac303e55a4087d2914bc240a80db67f',
        tool_calls=[],
        invalid_tool_calls=[],
        tool_call_chunks=[]
    ),
    {
        'ls_integration': 'langchain_chat_model',
        'langgraph_step': 1,
        'langgraph_node': 'model',
        'langgraph_triggers': ('branch:to:model',),
        'langgraph_path': ('__pregel_pull', 'model'),
        'langgraph_checkpoint_ns': 'model:7e2990ef-690a-caa5-9781-b156dd984445',
        'checkpoint_ns': 'model:7e2990ef-690a-caa5-9781-b156dd984445',
        'ls_provider': 'openai',
        'ls_model_name': 'gpt-6-luna',
        'ls_model_type': 'chat',
        'ls_temperature': None,
        'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2', 'langchain-openai': '1.6.4'}
    }
)

--------------------------------------------------


(
    AIMessageChunk(
        content=[
            {
                'type': 'function_call',
                'name': 'query_customer_data',
                'arguments': '',
                'call_id': 'call_3tz4Bt7ZeNl8fcDJsBSkRjFu',
                'id': 'fc_0f528ac9bcba3512016ac303e7a98487d2b1bbd063c2a4eeb3',
                'index': 0
            }
        ],
        additional_kwargs={},
        response_metadata={'model_provider': 'openai'},
        id='lc_run--01a109c7-2b4b-7830-8620-7d185818cae3',
        tool_calls=[
            {
                'name': 'query_customer_data',
                'args': {},
                'id': 'call_3tz4Bt7ZeNl8fcDJsBSkRjFu',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        tool_call_chunks=[
            {
                'name': 'query_customer_data',
                'args': '',
                'id': 'call_3tz4Bt7ZeNl8fcDJsBSkRjFu',
                'index': 0,
                'type': 'tool_call_chunk'
            }
        ]
    ),
    {
        'ls_integration': 'langchain_chat_model',
        'langgraph_step': 1,
        'langgraph_node': 'model',
        'langgraph_triggers': ('branch:to:model',),
        'langgraph_path': ('__pregel_pull', 'model'),
        'langgraph_checkpoint_ns': 'model:7e2990ef-690a-caa5-9781-b156dd984445',
        'checkpoint_ns': 'model:7e2990ef-690a-caa5-9781-b156dd984445',
        'ls_provider': 'openai',
        'ls_model_name': 'gpt-6-luna',
        'ls_model_type': 'chat',
        'ls_temperature': None,
        'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2', 'langchain-openai': '1.6.4'}
    }
)

--------------------------------------------------


(
    AIMessageChunk(
        content=[{'type': 'function_call', 'arguments': '{"customer_id":"CUST123456"}', 'index': 0}],
        additional_kwargs={},
        response_metadata={'model_provider': 'openai'},
        id='lc_run--01a109c7-2b4b-7830-8620-7d185818cae3',
        tool_calls=[{'name': '', 'args': {'customer_id': 'CUST123456'}, 'id': None, 'type': 'tool_call'}],
        invalid_tool_calls=[],
        tool_call_chunks=[
            {
                'name': None,
                'args': '{"customer_id":"CUST123456"}',
                'id': None,
                'index': 0,
                'type': 'tool_call_chunk'
            }
        ]
    ),
    {
        'ls_integration': 'langchain_chat_model',
        'langgraph_step': 1,
        'langgraph_node': 'model',
        'langgraph_triggers': ('branch:to:model',),
        'langgraph_path': ('__pregel_pull', 'model'),
        'langgraph_checkpoint_ns': 'model:7e2990ef-690a-caa5-9781-b156dd984445',
        'checkpoint_ns': 'model:7e2990ef-690a-caa5-9781-b156dd984445',
        'ls_provider': 'openai',
        'ls_model_name': 'gpt-6-luna',
        'ls_model_type': 'chat',
        'ls_temperature': None,
        'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2', 'langchain-openai': '1.6.4'}
    }
)

...

--------------------------------------------------


(
    AIMessageChunk(
        content=[
            {
                'type': 'text',
                'text': '',
                'id': 'msg_0f528ac9bcba3512016ac303eb4cf887d28b026a68d29f5ad6',
                'index': 0
            }
        ],
        additional_kwargs={},
        response_metadata={'model_provider': 'openai'},
        id='lc_run--01a109c7-43aa-7fa2-ba56-143a395efe99',
        tool_calls=[],
        invalid_tool_calls=[],
        tool_call_chunks=[]
    ),
    {
        'ls_integration': 'langchain_chat_model',
        'langgraph_step': 3,
        'langgraph_node': 'model',
        'langgraph_triggers': ('branch:to:model',),
        'langgraph_path': ('__pregel_pull', 'model'),
        'langgraph_checkpoint_ns': 'model:e0e55adb-850d-d8c0-9593-9bbd713fd884',
        'checkpoint_ns': 'model:e0e55adb-850d-d8c0-9593-9bbd713fd884',
        'ls_provider': 'openai',
        'ls_model_name': 'gpt-6-luna',
        'ls_model_type': 'chat',
        'ls_temperature': None,
        'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2', 'langchain-openai': '1.6.4'}
    }
)

--------------------------------------------------


(
    AIMessageChunk(
        content=[],
        additional_kwargs={},
        response_metadata={
            'created_at': 1791165418.0,
            'metadata': {},
            'model': 'gpt-6-luna',
            'object': 'response',
            'service_tier': 'default',
            'status': 'completed',
            'model_provider': 'openai',
            'model_name': 'gpt-6-luna'
        },
        id='lc_run--01a109c7-43aa-7fa2-ba56-143a395efe99',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 4750,
            'output_tokens': 91,
            'total_tokens': 4841,
            'input_token_details': {'cache_creation': 0, 'cache_read': 3584},
            'output_token_details': {'reasoning': 0}
        },
        tool_call_chunks=[],
        chunk_position='last'
    ),
    {
        'ls_integration': 'langchain_chat_model',
        'langgraph_step': 3,
        'langgraph_node': 'model',
        'langgraph_triggers': ('branch:to:model',),
        'langgraph_path': ('__pregel_pull', 'model'),
        'langgraph_checkpoint_ns': 'model:e0e55adb-850d-d8c0-9593-9bbd713fd884',
        'checkpoint_ns': 'model:e0e55adb-850d-d8c0-9593-9bbd713fd884',
        'ls_provider': 'openai',
        'ls_model_name': 'gpt-6-luna',
        'ls_model_type': 'chat',
        'ls_temperature': None,
        'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2', 'langchain-openai': '1.6.4'}
    }
)

--------------------------------------------------


### 8.2.4 tasks输出模式
任务开始和结束的通知。 Agent 的节点（model、tools）每运行一次就是一个"任务"。任务开始时输出一条，带有 input；结束时再输出一条，带有 result 和 error。你那次运行中模型同时调用了 3 个工具，所以出现了 3 个并行的 tools 任务。

In [8]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},stream_mode="tasks"
):
    rprint(chunk)
    print("-" * 50)

{
    'id': '8b93ac72-9525-42f3-f731-9330015bf14b',
    'name': 'model',
    'input': {
        'messages': [
            HumanMessage(
                content='查询客户ID为 CUST123456 的完整信息和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='fe28f1fb-0296-46d5-8462-9f14af4460c7'
            )
        ]
    },
    'triggers': ('branch:to:model',),
    'metadata': {'ls_integration': 'langchain_create_agent'}
}

--------------------------------------------------


{
    'id': '8b93ac72-9525-42f3-f731-9330015bf14b',
    'name': 'model',
    'error': None,
    'result': {
        'messages': [
            AIMessage(
                content=[
                    {
                        'arguments': '{"customer_id":"CUST123456"}',
                        'call_id': 'call_Ar5QgLPpbHWwzcDkNIBjdmjH',
                        'name': 'query_customer_data',
                        'type': 'function_call',
                        'id': 'fc_0f7bfa6d4338c634016ac305dc0a6087d2bd9d784723981a54',
                        'status': 'completed'
                    },
                    {
                        'arguments': '{"customer_id":"CUST123456"}',
                        'call_id': 'call_jmbsaEbltrN6QvJDelwLneJr',
                        'name': 'check_order_history',
                        'type': 'function_call',
                        'id': 'fc_0f7bfa6d4338c634016ac305dc0a7887d2b59461b027365ece',
                        'status': 'completed'
                    },
                    {
                        'arguments': '{}',
                        'call_id': 'call_LKRZaKbvkaZwXbpcitzJq70B',
                        'name': 'get_current_promotions',
                        'type': 'function_call',
                        'id': 'fc_0f7bfa6d4338c634016ac305dc0a8087d2b9f8dd8dcfb93b24',
                        'status': 'completed'
                    }
                ],
                additional_kwargs={},
                response_metadata={
                    'id': 'resp_0f7bfa6d4338c634016ac305da477887d2b5c71ff3f854b1fb',
                    'created_at': 1791165914.0,
                    'metadata': {},
                    'model': 'gpt-6-luna',
                    'object': 'response',
                    'service_tier': 'default',
                    'status': 'completed',
                    'model_provider': 'openai',
                    'model_name': 'gpt-6-luna'
                },
                id='resp_0f7bfa6d4338c634016ac305da477887d2b5c71ff3f854b1fb',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_Ar5QgLPpbHWwzcDkNIBjdmjH',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_jmbsaEbltrN6QvJDelwLneJr',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_LKRZaKbvkaZwXbpcitzJq70B',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 4574,
                    'output_tokens': 72,
                    'total_tokens': 4646,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                    'output_token_details': {'reasoning': 0}
                }
            )
        ]
    },
    'interrupts': []
}

--------------------------------------------------


{
    'id': '3181b631-0659-068e-22b3-3748f5d91c86',
    'name': 'tools',
    'input': [
        {
            'name': 'query_customer_data',
            'args': {'customer_id': 'CUST123456'},
            'id': 'call_Ar5QgLPpbHWwzcDkNIBjdmjH',
            'type': 'tool_call'
        }
    ],
    'triggers': ('__pregel_push',),
    'metadata': {'ls_integration': 'langchain_create_agent'}
}

...

--------------------------------------------------


{
    'id': 'b2d0f041-e6c7-6cc1-2b95-c91a8ec18467',
    'name': 'model',
    'input': {
        'messages': [
            HumanMessage(
                content='查询客户ID为 CUST123456 的完整信息和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='fe28f1fb-0296-46d5-8462-9f14af4460c7'
            ),
            AIMessage(
                content=[
                    {
                        'arguments': '{"customer_id":"CUST123456"}',
                        'call_id': 'call_Ar5QgLPpbHWwzcDkNIBjdmjH',
                        'name': 'query_customer_data',
                        'type': 'function_call',
                        'id': 'fc_0f7bfa6d4338c634016ac305dc0a6087d2bd9d784723981a54',
                        'status': 'completed'
                    },
                    {
                        'arguments': '{"customer_id":"CUST123456"}',
                        'call_id': 'call_jmbsaEbltrN6QvJDelwLneJr',
                        'name': 'check_order_history',
                        'type': 'function_call',
                        'id': 'fc_0f7bfa6d4338c634016ac305dc0a7887d2b59461b027365ece',
                        'status': 'completed'
                    },
                    {
                        'arguments': '{}',
                        'call_id': 'call_LKRZaKbvkaZwXbpcitzJq70B',
                        'name': 'get_current_promotions',
                        'type': 'function_call',
                        'id': 'fc_0f7bfa6d4338c634016ac305dc0a8087d2b9f8dd8dcfb93b24',
                        'status': 'completed'
                    }
                ],
                additional_kwargs={},
                response_metadata={
                    'id': 'resp_0f7bfa6d4338c634016ac305da477887d2b5c71ff3f854b1fb',
                    'created_at': 1791165914.0,
                    'metadata': {},
                    'model': 'gpt-6-luna',
                    'object': 'response',
                    'service_tier': 'default',
                    'status': 'completed',
                    'model_provider': 'openai',
                    'model_name': 'gpt-6-luna'
                },
                id='resp_0f7bfa6d4338c634016ac305da477887d2b5c71ff3f854b1fb',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_Ar5QgLPpbHWwzcDkNIBjdmjH',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_jmbsaEbltrN6QvJDelwLneJr',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_LKRZaKbvkaZwXbpcitzJq70B',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 4574,
                    'output_tokens': 72,
                    'total_tokens': 4646,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                    'output_token_details': {'reasoning': 0}
                }
            ),
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='25ee476a-caab-4066-bb90-074bf4294d0b',
                tool_call_id='call_Ar5QgLPpbHWwzcDkNIBjdmjH'
            ),
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='36c95956-1380-42a2-9dab-593c82ae1521',
                tool_call_id='call_jmbsaEbltrN6QvJD

--------------------------------------------------


{
    'id': 'b2d0f041-e6c7-6cc1-2b95-c91a8ec18467',
    'name': 'model',
    'error': None,
    'result': {
        'messages': [
            AIMessage(
                content=[
                    {
                        'type': 'text',
                        'text': '客户 CUST123456 的信息如下：\n\n- 姓名：张三\n- 等级：VIP\n- 
加入日期：2023-01-15\n- 历史订单数：15\n- 累计消费：25800.0\n\n当前可用优惠：老用户优惠、会员专属折扣，有效期至 
2027-01-31。',
                        'annotations': [],
                        'id': 'msg_0f7bfa6d4338c634016ac305dfb8d487d2ade56483e196b817',
                        'phase': 'final_answer'
                    }
                ],
                additional_kwargs={},
                response_metadata={
                    'id': 'resp_0f7bfa6d4338c634016ac305de874887d2a56df7de00490163',
                    'created_at': 1791165918.0,
                    'metadata': {},
                    'model': 'gpt-6-luna',
                    'object': 'response',
                    'service_tier': 'default',
                    'status': 'completed',
                    'model_provider': 'openai',
                    'model_name': 'gpt-6-luna'
                },
                id='resp_0f7bfa6d4338c634016ac305de874887d2a56df7de00490163',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 4750,
                    'output_tokens': 85,
                    'total_tokens': 4835,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                    'output_token_details': {'reasoning': 0}
                }
            )
        ]
    },
    'interrupts': []
}

--------------------------------------------------


**读懂 tasks 的输出**

先说"任务"（task）是什么。01 笔记讲过，Agent 是一张图，由 `model`、`tools` 等节点组成。**节点每运行一次，就是一个任务**。`tasks` 模式在每个任务**开始**和**结束**时各输出一条事件：

| 事件 | 字段 | 含义 |
|---|---|---|
| 开始 | `id` | 任务编号。开始和结束两条事件的 `id` 相同，可以据此配对 |
| | `name` | 节点名：`model` 或 `tools` |
| | `input` | 节点收到的输入：`model` 收到的是消息列表，`tools` 收到的是要执行的工具调用 |
| | `triggers` | 这个任务是由什么触发的，一般不用关心 |
| 结束 | `result` | 节点返回的结果，和 `updates` 模式看到的更新是同一份数据 |
| | `error` | 出错时的异常信息，正常时为 `None` |
| | `interrupts` | 中断信息，人工审批等场景才会用到 |

上面一共 10 条事件：`model` 开始、结束 → 3 个 `tools` 开始 → 3 个 `tools` 结束 → `model` 开始、结束。模型一次要求调用三个工具，`create_agent` 会**为每个工具调用单独起一个 `tools` 任务、同时执行**，所以每个 `tools` 任务的 `input` 里只有一个工具调用，三个开始事件挨在一起，三个结束事件也挨在一起。

注意：`tasks` 事件里**没有时间**。想看时间戳，要用下面的 `debug` 模式。

### 8.2.5 debug输出模式

该模式与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式用于调试、监控task任务的生命周期。

但实质是tasks 加 checkpoints，再加上步数和时间戳。 源码里它就是把这两种模式的事件各包一层 {step, timestamp, type, payload} 外壳。你的输出里只有 task 和 task_result，因为那时的 Agent 还没有设置 checkpointer。

In [9]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},stream_mode="debug"
):
    rprint(chunk)
    print("-" * 50)

{
    'step': 1,
    'timestamp': '2026-10-05T02:18:25.357556+00:00',
    'type': 'task',
    'payload': {
        'id': '140a5d49-3013-7763-7a44-aced65bc9eef',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='查询客户ID为 CUST123456 的完整信息和可用优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='17ffdb4c-b8fe-4fde-909b-615063031871'
                )
            ]
        },
        'triggers': ('branch:to:model',),
        'metadata': {'ls_integration': 'langchain_create_agent'}
    }
}

--------------------------------------------------


{
    'step': 1,
    'timestamp': '2026-10-05T02:18:33.039348+00:00',
    'type': 'task_result',
    'payload': {
        'id': '140a5d49-3013-7763-7a44-aced65bc9eef',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content=[
                        {
                            'arguments': '{"customer_id":"CUST123456"}',
                            'call_id': 'call_DN2hVpEhx3r30ti8EXFLd6D4',
                            'name': 'query_customer_data',
                            'type': 'function_call',
                            'id': 'fc_0f5adf2e97cb900c016ac308f851f887d2bcfa135b0b335a2d',
                            'status': 'completed'
                        }
                    ],
                    additional_kwargs={},
                    response_metadata={
                        'id': 'resp_0f5adf2e97cb900c016ac308f74fe087d2bf5c077fb820405f',
                        'created_at': 1791166711.0,
                        'metadata': {},
                        'model': 'gpt-6-luna',
                        'object': 'response',
                        'service_tier': 'default',
                        'status': 'completed',
                        'model_provider': 'openai',
                        'model_name': 'gpt-6-luna'
                    },
                    id='resp_0f5adf2e97cb900c016ac308f74fe087d2bf5c077fb820405f',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'CUST123456'},
                            'id': 'call_DN2hVpEhx3r30ti8EXFLd6D4',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 4574,
                        'output_tokens': 23,
                        'total_tokens': 4597,
                        'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                        'output_token_details': {'reasoning': 0}
                    }
                )
            ]
        },
        'interrupts': []
    }
}

--------------------------------------------------


{
    'step': 2,
    'timestamp': '2026-10-05T02:18:33.039656+00:00',
    'type': 'task',
    'payload': {
        'id': '1d3d1a7f-01a4-7e05-8a74-8c4d4da93641',
        'name': 'tools',
        'input': [
            {
                'name': 'query_customer_data',
                'args': {'customer_id': 'CUST123456'},
                'id': 'call_DN2hVpEhx3r30ti8EXFLd6D4',
                'type': 'tool_call'
            }
        ],
        'triggers': ('__pregel_push',),
        'metadata': {'ls_integration': 'langchain_create_agent'}
    }
}

...

--------------------------------------------------


{
    'step': 7,
    'timestamp': '2026-10-05T02:18:45.791031+00:00',
    'type': 'task',
    'payload': {
        'id': '17d1e287-247b-0e5c-db8d-624ed5326e1d',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='查询客户ID为 CUST123456 的完整信息和可用优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='17ffdb4c-b8fe-4fde-909b-615063031871'
                ),
                AIMessage(
                    content=[
                        {
                            'arguments': '{"customer_id":"CUST123456"}',
                            'call_id': 'call_DN2hVpEhx3r30ti8EXFLd6D4',
                            'name': 'query_customer_data',
                            'type': 'function_call',
                            'id': 'fc_0f5adf2e97cb900c016ac308f851f887d2bcfa135b0b335a2d',
                            'status': 'completed'
                        }
                    ],
                    additional_kwargs={},
                    response_metadata={
                        'id': 'resp_0f5adf2e97cb900c016ac308f74fe087d2bf5c077fb820405f',
                        'created_at': 1791166711.0,
                        'metadata': {},
                        'model': 'gpt-6-luna',
                        'object': 'response',
                        'service_tier': 'default',
                        'status': 'completed',
                        'model_provider': 'openai',
                        'model_name': 'gpt-6-luna'
                    },
                    id='resp_0f5adf2e97cb900c016ac308f74fe087d2bf5c077fb820405f',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'CUST123456'},
                            'id': 'call_DN2hVpEhx3r30ti8EXFLd6D4',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 4574,
                        'output_tokens': 23,
                        'total_tokens': 4597,
                        'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                        'output_token_details': {'reasoning': 0}
                    }
                ),
                ToolMessage(
                    content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                    name='query_customer_data',
                    id='e41d7582-2721-416e-afb6-bb71772c8b40',
                    tool_call_id='call_DN2hVpEhx3r30ti8EXFLd6D4'
                ),
                AIMessage(
                    content=[
                        {
                            'arguments': '{"customer_id":"CUST123456"}',
                            'call_id': 'call_pmoIMjoDjebkBr03tOgRqofR',
                            'name': 'check_order_history',
                            'type': 'function_call',
                            'id': 'fc_0f5adf2e97cb900c016ac308ff8d1487d28fbb68037e2afc87',
                            'status': 'completed'
                        }
                    ],
                    additional_kwargs={},
                    response_metadata={
                        'id': 'resp_0f5adf2e97cb900c016ac308fe450087d29a3aad97a9d42db5',
                        'created_at': 1791166718.0,
                        'metadata': {},
                        'model': 'gpt-6-luna',
                        'object': 'response',
                        'service_tier': 'default',
                        'status': 'completed',
                        'model_provider': 'openai',
                        'model_name': 'gpt-6-luna'
                    },
                    id='resp_0f5adf2e97cb900c016ac308fe450087d29a3aad97a9d42db5',
                    tool_calls=[
                        

--------------------------------------------------


{
    'step': 7,
    'timestamp': '2026-10-05T02:18:49.511386+00:00',
    'type': 'task_result',
    'payload': {
        'id': '17d1e287-247b-0e5c-db8d-624ed5326e1d',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content=[
                        {
                            'type': 'text',
                            'text': '客户 CUST123456 的信息：\n\n- 姓名：张三\n- 等级：VIP\n- 
加入日期：2023-01-15\n- 总订单数：15\n- 总消费：25,800.00\n\n当前可用优惠：老用户优惠、会员专属折扣，有效期至 
2027-01-31。',
                            'annotations': [],
                            'id': 'msg_0f5adf2e97cb900c016ac3090780e887d2a263eb094fde136f',
                            'phase': 'final_answer'
                        }
                    ],
                    additional_kwargs={},
                    response_metadata={
                        'id': 'resp_0f5adf2e97cb900c016ac30906fa5c87d2bd39ea3f474d1dae',
                        'created_at': 1791166727.0,
                        'metadata': {},
                        'model': 'gpt-6-luna',
                        'object': 'response',
                        'service_tier': 'default',
                        'status': 'completed',
                        'model_provider': 'openai',
                        'model_name': 'gpt-6-luna'
                    },
                    id='resp_0f5adf2e97cb900c016ac30906fa5c87d2bd39ea3f474d1dae',
                    tool_calls=[],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 4744,
                        'output_tokens': 81,
                        'total_tokens': 4825,
                        'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                        'output_token_details': {'reasoning': 0}
                    }
                )
            ]
        },
        'interrupts': []
    }
}

--------------------------------------------------


**读懂 debug 的输出**

`debug` 并不是一种新的事件。源码（`langgraph/pregel/_loop.py`）里，它就是把 `tasks` 和 `checkpoints` 两种模式的事件，**各包一层外壳**再输出：

```python
{
    "step": ...,        # 第几步
    "timestamp": ...,   # 输出这条事件的时间（UTC 时区）
    "type": ...,        # "task"：任务开始；"task_result"：任务结束；"checkpoint"：保存了状态快照
    "payload": ...,     # 原始事件，也就是 tasks 或 checkpoints 模式输出的那个字典
}
```

上面的输出里只有 `task` 和 `task_result` 两种类型，没有 `checkpoint`，因为这时的 Agent 还没有设置 `checkpointer`，不会保存快照（见 8.2.6）。这次模型是一个一个地调用工具的，所以一共 7 步（`model` 和 `tools` 交替 3 轮，最后再运行一次 `model`），每步一对开始、结束事件，共 14 条。

所以可以记成：**debug = tasks + checkpoints + 步数和时间戳**。它的信息最全，也最啰嗦，适合排查问题时使用。

### 8.2.6 checkpoints输出模式
该模式中，每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景。

状态存档。 Agent 每走完一步，就把完整状态保存一份，内容包括会话编号、存档编号、完整状态，以及下一步要去哪个节点。必须设置 checkpointer 才会有输出，否则一条都没有。你那次运行共有 5 个存档，笔记里有一张表逐个说明了每个存档对应发生了什么。

In [12]:
from langgraph.checkpoint.memory import InMemorySaver
# 1. 创建内存检查点存储
checkpointer = InMemorySaver()

customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions],
    checkpointer=checkpointer
)
# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用Agent
checkpoint_count = 0

for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},config=config,stream_mode="checkpoints"):
    checkpoint_count += 1
    print(f"Checkpoint # {checkpoint_count}:")
    rprint(chunk)
    print("-" * 50)

Checkpoint # 1:


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1c0649-58e8-684a-bfff-47d7a1563d16'
        }
    },
    'parent_config': None,
    'values': {'messages': []},
    'metadata': {'source': 'input', 'step': -1, 'parents': {}},
    'next': ['__start__'],
    'tasks': [
        {'id': 'e916c07a-f967-9be1-cde0-2127559eae2e', 'name': '__start__', 'interrupts': (), 'state': None}
    ]
}

--------------------------------------------------
Checkpoint # 2:


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1c0649-58f1-66ac-8000-13f7ab9e7efa'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1c0649-58e8-684a-bfff-47d7a1563d16'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content='查询客户ID为 CUST123456 的完整信息和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='6bdcdaaa-d5bf-4148-9dc7-69a0396913d3'
            )
        ]
    },
    'metadata': {'source': 'loop', 'step': 0, 'parents': {}},
    'next': ['model'],
    'tasks': [{'id': '4945ff36-bf16-5094-a265-d6a0d1bf4178', 'name': 'model', 'interrupts': (), 'state': None}]
}

--------------------------------------------------
Checkpoint # 3:


...

--------------------------------------------------
Checkpoint # 4:


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1c0649-a04c-64b6-8002-6f3bc5aee7b4'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1c0649-a027-63fa-8001-73f27ecd7ca7'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content='查询客户ID为 CUST123456 的完整信息和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='6bdcdaaa-d5bf-4148-9dc7-69a0396913d3'
            ),
            AIMessage(
                content=[
                    {
                        'arguments': '{"customer_id":"CUST123456"}',
                        'call_id': 'call_lsGN92bdyDJ0zWvHX1EeqOkC',
                        'name': 'query_customer_data',
                        'type': 'function_call',
                        'id': 'fc_0e616a5d8bccdd0d016ac30b8d634487d28730fc2f2b179e99',
                        'status': 'completed'
                    },
                    {
                        'arguments': '{"customer_id":"CUST123456"}',
                        'call_id': 'call_ERKxoizljMwBnWUUkZq6e07W',
                        'name': 'check_order_history',
                        'type': 'function_call',
                        'id': 'fc_0e616a5d8bccdd0d016ac30b8d635487d2849534692f769e17',
                        'status': 'completed'
                    },
                    {
                        'arguments': '{}',
                        'call_id': 'call_ESMEEQRGJMSokDi9WFsBzSqe',
                        'name': 'get_current_promotions',
                        'type': 'function_call',
                        'id': 'fc_0e616a5d8bccdd0d016ac30b8d635c87d2bea00abc736acca8',
                        'status': 'completed'
                    }
                ],
                additional_kwargs={},
                response_metadata={
                    'id': 'resp_0e616a5d8bccdd0d016ac30b89431487d2845635e5b07d2d86',
                    'created_at': 1791167369.0,
                    'metadata': {},
                    'model': 'gpt-6-luna',
                    'object': 'response',
                    'service_tier': 'default',
                    'status': 'completed',
                    'model_provider': 'openai',
                    'model_name': 'gpt-6-luna'
                },
                id='resp_0e616a5d8bccdd0d016ac30b89431487d2845635e5b07d2d86',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_lsGN92bdyDJ0zWvHX1EeqOkC',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_ERKxoizljMwBnWUUkZq6e07W',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_ESMEEQRGJMSokDi9WFsBzSqe',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 4574,
                    'output_tokens': 72,
                    'total_tokens': 4646,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 3584},
                    'output_token_details': {'reasoning': 0}
                }
            ),
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='f210c4dd-a823-4307-bfd6-a8cf5c

--------------------------------------------------
Checkpoint # 5:


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1c0649-d924-6d60-8003-8d32809d51be'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1c0649-a04c-64b6-8002-6f3bc5aee7b4'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content='查询客户ID为 CUST123456 的完整信息和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='6bdcdaaa-d5bf-4148-9dc7-69a0396913d3'
            ),
            AIMessage(
                content=[
                    {
                        'arguments': '{"customer_id":"CUST123456"}',
                        'call_id': 'call_lsGN92bdyDJ0zWvHX1EeqOkC',
                        'name': 'query_customer_data',
                        'type': 'function_call',
                        'id': 'fc_0e616a5d8bccdd0d016ac30b8d634487d28730fc2f2b179e99',
                        'status': 'completed'
                    },
                    {
                        'arguments': '{"customer_id":"CUST123456"}',
                        'call_id': 'call_ERKxoizljMwBnWUUkZq6e07W',
                        'name': 'check_order_history',
                        'type': 'function_call',
                        'id': 'fc_0e616a5d8bccdd0d016ac30b8d635487d2849534692f769e17',
                        'status': 'completed'
                    },
                    {
                        'arguments': '{}',
                        'call_id': 'call_ESMEEQRGJMSokDi9WFsBzSqe',
                        'name': 'get_current_promotions',
                        'type': 'function_call',
                        'id': 'fc_0e616a5d8bccdd0d016ac30b8d635c87d2bea00abc736acca8',
                        'status': 'completed'
                    }
                ],
                additional_kwargs={},
                response_metadata={
                    'id': 'resp_0e616a5d8bccdd0d016ac30b89431487d2845635e5b07d2d86',
                    'created_at': 1791167369.0,
                    'metadata': {},
                    'model': 'gpt-6-luna',
                    'object': 'response',
                    'service_tier': 'default',
                    'status': 'completed',
                    'model_provider': 'openai',
                    'model_name': 'gpt-6-luna'
                },
                id='resp_0e616a5d8bccdd0d016ac30b89431487d2845635e5b07d2d86',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_lsGN92bdyDJ0zWvHX1EeqOkC',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_ERKxoizljMwBnWUUkZq6e07W',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_ESMEEQRGJMSokDi9WFsBzSqe',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 4574,
                    'output_tokens': 72,
                    'total_tokens': 4646,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 3584},
                    'output_token_details': {'reasoning': 0}
                }
            ),
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='f210c4dd-a823-4307-bfd6-a8cf5c

--------------------------------------------------


**读懂 checkpoints 的输出**

**检查点**（checkpoint）就是状态的"存档"。设置了 `checkpointer` 后，Agent 每走完一步，就把当时的完整状态保存一份，01 笔记 2.5.5 的多轮记忆靠的就是它。`checkpoints` 模式在**每次存档时**输出一条事件，格式和 `agent.get_state(config)` 的返回值相同：

| 字段 | 含义 |
|---|---|
| `config` | 这个存档的地址：`thread_id`（会话编号）加上 `checkpoint_id`（存档编号） |
| `parent_config` | 上一个存档的地址，第一个存档为 `None` |
| `values` | 存档时的完整状态，和 `values` 模式输出的内容一样 |
| `metadata` | `source` 为 `input` 表示刚收到输入，为 `loop` 表示循环中的某一步；`step` 是步数，从 -1 开始 |
| `next` | 下一步要运行哪些节点，空列表表示运行结束 |
| `tasks` | 下一步要执行的任务 |

上面一共 5 个存档：

| 存档 | `step` | `next` | 这时发生了什么 |
|---|---|---|---|
| 1 | -1 | `__start__` | 刚收到输入，状态里的消息列表还是空的 |
| 2 | 0 | `model` | 用户的消息已经写入状态 |
| 3 | 1 | `tools` ×3 | 模型返回了三个工具调用 |
| 4 | 2 | `model` | 三个工具的结果已经写入状态 |
| 5 | 3 | （空） | 模型给出了最终回答，运行结束 |

两点注意：

- **必须设置 `checkpointer` 才会有输出**。没有它就没有存档，`checkpoints` 模式一条事件都不会输出
- 这些存档还能用来"回到过去"：用 `agent.get_state_history(config)` 查看全部存档，或者从某个存档重新运行。这属于 LangGraph 的进阶用法

### 8.2.7 custom输出模式
开发者通过 get_stream_writer 在工具或节点内部 自定义发送的数据 ，用于 输出 业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标。

举例：生成销售报告和库存报告Agent

In [13]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time

@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()
    writer({"type": "生成销售报告", "message": "开始生成销售报告"})
    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告","message": f"生成销售报告进度百分比：{i * 25}%"})
    writer({"type": "生成销售报告", "message": "报告生成完成"})
    return f"销售报告：总收入150万元，同比增长12%"

@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")
    return "当前库存量为10000件，库存充足，无异常"


# 创建报告生成agent
reporting_agent = create_agent(
    model=model,
    tools=[generate_sales_report, generate_inventory_report]
)

for chunk in reporting_agent.stream(
    {"messages": [{"role": "user","content": "生成销售报告和库存报告"}]},
    stream_mode="custom"
):
    print(chunk)
    print("-" * 50)

{'type': '生成销售报告', 'message': '开始生成销售报告'}
--------------------------------------------------
开始库存分析...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：25%'}
--------------------------------------------------
检查当前库存量...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：50%'}
--------------------------------------------------
生成库存报告...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：75%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '报告生成完成'}
--------------------------------------------------


**读懂 custom 的输出**

前面几种模式输出的都是 LangGraph 自己产生的数据，`custom` 模式输出的则是**你自己发出的数据**：

1. 在工具（或节点）里调用 `get_stream_writer()`，拿到一个"写入函数" `writer`
2. 调用 `writer(数据)`，这份数据会立即作为一条事件输出。`writer` 收到什么，`stream` 就输出什么：上面 `generate_sales_report` 发的是字典，`generate_inventory_report` 发的是字符串

几个现象和注意点：

- **两份报告的进度交替出现**：模型一次调用了两个工具，两个工具是同时执行的（见 tasks 部分），所以它们发出的进度混在了一起。在前端展示时，最好在数据里标明来源，像 `generate_sales_report` 那样放一个 `type` 字段
- **只输出自定义数据**：`custom` 模式不包括模型的回复。想同时看到进度和回复，可以同时开启多种模式，见 8.3
- **不开 custom 模式也不会报错**：这时 `writer(...)` 什么都不做，所以工具里的这些代码可以一直保留

## 8.3 把几种模式放在一起看：一次运行的时间线

`stream_mode` 也可以传一个列表，同时开启多种模式。这时每条输出变成一个元组 `(模式名, 数据)`，可以看出各种事件在一次运行里的先后顺序。

下面用一个简单的天气 Agent（工具里用 `writer` 发一条进度），同时开启 `updates`、`tasks`、`checkpoints`、`custom` 四种模式，每条事件只打印一行摘要（会请求模型两次左右）：

In [14]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer

@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气"""
    writer = get_stream_writer()
    writer(f"正在查询{city}的天气...")
    return f"{city}：晴，22 摄氏度"

timeline_agent = create_agent(model=model, tools=[get_weather], checkpointer=InMemorySaver())

def brief(mode, data):
    """把一条事件缩成一行摘要"""
    if mode == "checkpoints":
        return f"保存快照  step={data['metadata']['step']}  next={data['next']}  消息数={len(data['values'].get('messages', []))}"
    if mode == "tasks":
        return f"{'结束' if 'result' in data else '开始'} {data['name']} 任务"
    if mode == "updates":
        node, update = next(iter(data.items()))
        msg = update["messages"][-1]
        return f"{node} 节点返回 {type(msg).__name__}" + ("（工具调用）" if getattr(msg, "tool_calls", None) else "")
    if mode == "custom":
        return f"自定义数据：{data}"

config = {"configurable": {"thread_id": "timeline-demo"}}
for mode, data in timeline_agent.stream(
    {"messages": [{"role": "user", "content": "北京今天天气怎么样？"}]},
    config=config,
    stream_mode=["updates", "tasks", "checkpoints", "custom"],
):
    print(f"[{mode:<11}] {brief(mode, data)}")

[checkpoints] 保存快照  step=-1  next=['__start__']  消息数=0
[checkpoints] 保存快照  step=0  next=['model']  消息数=1
[tasks      ] 开始 model 任务


[updates    ] model 节点返回 AIMessage（工具调用）
[tasks      ] 结束 model 任务
[checkpoints] 保存快照  step=1  next=['tools']  消息数=2
[tasks      ] 开始 tools 任务
[custom     ] 自定义数据：正在查询北京的天气...
[updates    ] tools 节点返回 ToolMessage
[tasks      ] 结束 tools 任务
[checkpoints] 保存快照  step=2  next=['model']  消息数=3
[tasks      ] 开始 model 任务


[updates    ] model 节点返回 AIMessage
[tasks      ] 结束 model 任务
[checkpoints] 保存快照  step=3  next=[]  消息数=4


从时间线可以看出，每一步都是同样的节奏：

```text
保存快照(checkpoints) → 开始任务(tasks) → [工具运行时发出的自定义数据(custom)] → 节点返回更新(updates) → 结束任务(tasks) → 保存快照(checkpoints) → ……
```

- `checkpoints` 把每一步隔开：第一个快照（step=-1）保存的是刚收到的输入，最后一个快照（step=3）的 `next` 为空，表示运行结束
- `tasks` 的开始、结束事件包住了节点的运行过程；`updates` 出现在节点结束之前，内容就是 `tasks` 结束事件里的 `result`
- `custom` 只会出现在工具（或节点）运行的过程中，也就是 `tools` 任务的开始和结束之间
- 没有开启的几种模式也能放进这条时间线：`values` 输出的时机和 `checkpoints` 差不多，内容就是快照里的 `values`；`messages` 的片段会在 `model` 任务的开始和结束之间一段一段地出现；`debug` 就是 `tasks` 和 `checkpoints` 这两种事件，再加上步数和时间戳

## 8.4 对比总结

| 模式 | 输出什么 | 什么时候输出 | 每条数据的样子 | 典型用途 |
|---|---|---|---|---|
| `values` | 完整状态 | 每一步结束后 | `{"messages": [到目前为止的全部消息]}` | 随时需要完整的对话和状态 |
| `updates`（默认） | 这一步的变化 | 每个节点运行完 | `{"节点名": {"messages": [新消息]}}` | 展示进度：调用了哪个工具、工具返回了什么 |
| `messages` | 模型生成的内容片段 | 模型生成的过程中，一小段一小段地输出 | `(消息片段, 元数据)`，元数据里有 `langgraph_node` 等 | 聊天界面的打字机效果 |
| `tasks` | 任务的开始和结束 | 每个节点开始时、结束时各一次 | 开始：`{id, name, input, triggers}`；结束：`{id, name, result, error, interrupts}` | 监控每个节点的输入、输出和错误 |
| `checkpoints` | 状态存档 | 每次保存检查点（需要 `checkpointer`） | `{config, parent_config, values, metadata, next, tasks}` | 持久化、断点恢复、查看历史存档 |
| `debug` | `tasks` 和 `checkpoints` 的全部事件 | 同 `tasks` 和 `checkpoints` | `{step, timestamp, type, payload}` | 调试，信息最全 |
| `custom` | 你自己发出的数据 | 调用 `writer(...)` 时 | `writer` 收到什么就是什么 | 业务进度条、自定义日志 |

**几组容易混淆的模式：**

- **`values`、`updates`、`checkpoints` 都和状态有关**：`values` 每步给完整状态；`updates` 只给这一步新增的部分；`checkpoints` 也给完整状态，但还带着存档编号、下一步去哪等信息，并且需要 `checkpointer`
- **`tasks` 和 `updates`**：`updates` 只在节点结束时输出它返回的数据；`tasks` 在节点开始时也会输出，能看到节点的输入，还能看到出错信息
- **`debug` 和 `tasks`、`checkpoints`**：`debug` 就是后两者的事件加上步数和时间戳

**怎么选：**

- 做聊天界面：`messages` 负责打字机效果，`updates` 显示调用了哪个工具，`custom` 显示工具的进度。实际项目里经常同时开启：`stream_mode=["messages", "updates", "custom"]`
- 只想知道 Agent 走到哪一步了：`updates`
- 排查问题：`debug`
- 需要存档或恢复运行：先设置 `checkpointer`，再用 `checkpoints`